# OOF Evaluation — Diagnostics and Cohort Analysis


## 1. Imports and Configuration


In [ ]:
# Core libraries
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

MODEL_ID = 'xgboost'
RUN_NAME = "nusc_mini_debug_tpp-11_Mar_2026_15_29_02"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'

RESULTS_ROOT = Path('../../results/interpretable_model') / MODEL_ID / RUN_NAME
TABLES_DIR = RESULTS_ROOT / 'tables'
PLOTS_DIR = RESULTS_ROOT / 'plots'

print(f'Results root: {RESULTS_ROOT.resolve()}')
print(f'Model ID: {MODEL_ID}')
print(f'Run: {RUN_NAME}')
print(f'TARGET_COL override: {TARGET_COL}')


## 2. Load Run Manifest and Nested-Resampling Artifacts


In [ ]:
def resolve_manifest_path(model_id, run_name, target_col=None):
    manifest_dir = Path('../../results/interpretable_model') / model_id / run_name / 'tables'
    if not manifest_dir.exists():
        raise FileNotFoundError(f'No tables directory found for model_id={model_id}, run_name={run_name}: {manifest_dir}')

    if target_col is not None:
        manifest_path = manifest_dir / f'run_manifest_{target_col}.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'Run manifest not found for target_col={target_col}: {manifest_path}')
        return manifest_path

    manifest_candidates = sorted(manifest_dir.glob('run_manifest_*.json'))
    if not manifest_candidates:
        raise FileNotFoundError(f'No run_manifest_*.json files found in {manifest_dir}')
    if len(manifest_candidates) > 1:
        raise ValueError(
            f'Multiple run manifests found in {manifest_dir}. Set TARGET_COL explicitly. '
            f'Candidates: {[p.name for p in manifest_candidates]}'
        )
    return manifest_candidates[0]


manifest_path = resolve_manifest_path(MODEL_ID, RUN_NAME, TARGET_COL)
manifest = json.loads(manifest_path.read_text())

if manifest['model_id'] != MODEL_ID:
    raise ValueError(f"Manifest model_id={manifest['model_id']} does not match MODEL_ID={MODEL_ID}")

target_col = manifest['target_col']
feature_cols = manifest['feature_cols']
TABLES_DIR = Path(manifest['tables_dir'])
PLOTS_DIR = Path(manifest['plots_dir'])
POOR_WELL_QUANTILE = manifest['analysis'].get('poor_well_quantile', 0.20)

nested_resampling = manifest['nested_resampling']
model_data_path = Path(nested_resampling['model_data_with_oof_path'])
metrics_path = Path(nested_resampling['oof_metrics_path'])

model_df_oof = pd.read_csv(model_data_path)
oof_metrics_df = pd.read_csv(metrics_path)

X = model_df_oof[feature_cols]
y_oof_orig = model_df_oof['target_orig'].to_numpy()
y_oof_pred_orig = model_df_oof['oof_pred_orig'].to_numpy()

print(f'Loaded manifest: {manifest_path}')
print(f'Loaded model data: {model_data_path}')
print(f'Loaded OOF metrics: {metrics_path}')
print(f'Target: {target_col} | Features: {len(feature_cols)}')
display(oof_metrics_df)


## 3. Model-Fit Diagnostics (OOF)


In [ ]:
residuals_oof = y_oof_orig - y_oof_pred_orig

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(y_oof_orig, y_oof_pred_orig, alpha=0.35, s=18, color='steelblue')
lims = [
    min(np.min(y_oof_orig), np.min(y_oof_pred_orig)),
    max(np.max(y_oof_orig), np.max(y_oof_pred_orig)),
]
axes[0, 0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0, 0].set_xlabel(f'Actual {target_col} (original scale)')
axes[0, 0].set_ylabel(f'Predicted {target_col} (original scale)')
axes[0, 0].set_title('Predicted vs Actual (OOF)')
axes[0, 0].legend()

axes[0, 1].scatter(y_oof_pred_orig, residuals_oof, alpha=0.35, s=18, color='steelblue')
axes[0, 1].axhline(0.0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted (original scale)')
axes[0, 1].set_ylabel('Residuals (actual - predicted)')
axes[0, 1].set_title('Residuals vs Predicted (OOF)')

axes[1, 0].hist(residuals_oof, bins=40, density=True, alpha=0.75, color='steelblue', edgecolor='black')
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title(f'Residual Distribution (skew={stats.skew(residuals_oof):.3f})')

stats.probplot(residuals_oof, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals')

plt.suptitle(f'{MODEL_ID} Residual Diagnostics — Target: {target_col}', fontsize=15, y=1.02)
plt.tight_layout()
diag_path = PLOTS_DIR / f'residual_diagnostics_oof_{target_col}.png'
plt.savefig(diag_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Diagnostics plot saved to: {diag_path}')
print(f'Residual mean:   {residuals_oof.mean():.6f}')
print(f'Residual median: {np.median(residuals_oof):.6f}')
print(f'Residual std:    {residuals_oof.std():.6f}')


## 4. Cohort Analysis: Poor vs Well Performance (OOF)


In [ ]:
abs_error_oof = np.abs(y_oof_orig - y_oof_pred_orig)
poor_threshold = np.quantile(abs_error_oof, 1.0 - POOR_WELL_QUANTILE)
well_threshold = np.quantile(abs_error_oof, POOR_WELL_QUANTILE)

poor_mask = abs_error_oof >= poor_threshold
well_mask = abs_error_oof <= well_threshold

X_reset = X.reset_index(drop=True).copy()
X_reset['abs_error'] = abs_error_oof

poor_df = X_reset.loc[poor_mask].copy()
well_df = X_reset.loc[well_mask].copy()

print(f'Poor threshold (top {POOR_WELL_QUANTILE:.0%}): abs_error >= {poor_threshold:.6f}')
print(f'Well threshold (bottom {POOR_WELL_QUANTILE:.0%}): abs_error <= {well_threshold:.6f}')
print(f'Poor cohort size: {len(poor_df)}')
print(f'Well cohort size: {len(well_df)}')


In [ ]:
def standardized_difference(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    pooled = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2.0)
    if not np.isfinite(pooled) or pooled < eps:
        return 0.0
    return (np.mean(a) - np.mean(b)) / pooled

cohort_rows = []
for feat in X.columns:
    poor_vals = poor_df[feat].dropna().values
    well_vals = well_df[feat].dropna().values

    cohort_rows.append({
        'feature': feat,
        'poor_mean': np.mean(poor_vals),
        'well_mean': np.mean(well_vals),
        'poor_median': np.median(poor_vals),
        'well_median': np.median(well_vals),
        'mean_diff': np.mean(poor_vals) - np.mean(well_vals),
        'median_diff': np.median(poor_vals) - np.median(well_vals),
        'standardized_diff': standardized_difference(poor_vals, well_vals),
    })

cohort_df = pd.DataFrame(cohort_rows)
cohort_df['abs_standardized_diff'] = cohort_df['standardized_diff'].abs()
cohort_df = cohort_df.sort_values('abs_standardized_diff', ascending=False).reset_index(drop=True)

display(cohort_df)

cohort_path = TABLES_DIR / f'cohort_comparison_{target_col}.csv'
cohort_df.to_csv(cohort_path, index=False)
print(f'Cohort comparison table saved to: {cohort_path}')


In [ ]:
top_diff_features = cohort_df['feature'].head(6).tolist()
print('Top differentiating features (|standardized_diff|):')
print(top_diff_features)

n_plot = len(top_diff_features)
n_cols = 3
n_rows = int(np.ceil(n_plot / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, feat in enumerate(top_diff_features):
    long_df = pd.concat([
        pd.DataFrame({'cohort': f'Well (bottom {POOR_WELL_QUANTILE:.0%})', 'value': well_df[feat].values}),
        pd.DataFrame({'cohort': f'Poor (top {POOR_WELL_QUANTILE:.0%})', 'value': poor_df[feat].values}),
    ], ignore_index=True)

    sns.violinplot(data=long_df, x='cohort', y='value', ax=axes[i], inner='box', cut=0)
    axes[i].set_title(feat)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=20)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Poor vs Well Cohorts — Feature Distributions', fontsize=15, y=1.02)
plt.tight_layout()
cohort_plot_path = PLOTS_DIR / f'cohort_violin_top_features_{target_col}.png'
plt.savefig(cohort_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Cohort violin plots saved to: {cohort_plot_path}')


## 5. Sanity Checks and Saved Artifacts


In [ ]:
poor_median_error = np.median(abs_error_oof[poor_mask])
well_median_error = np.median(abs_error_oof[well_mask])
print(f'Poor median abs error: {poor_median_error:.6f}')
print(f'Well median abs error: {well_median_error:.6f}')
print(f'Check poor > well: {poor_median_error > well_median_error}')

print('\nSaved artifacts:')
print(f'- Run manifest:          {manifest_path}')
print(f'- OOF metrics table:     {metrics_path}')
print(f'- Cohort comparison:     {cohort_path}')
print(f'- Plot directory:        {PLOTS_DIR}')
